# Stores API Reference

Developer-facing statements defined in `libs/core/langchain_core/stores.py`.

# `BaseStore: ABC, Generic[K, V]`

Abstract batch-oriented interface for a key-value store.

Concrete subclasses must implement `mget`, `mset`, `mdelete`, and `yield_keys`. The asynchronous methods are optional wrappers that delegate to the synchronous methods through `run_in_executor`.

The interface accepts batches rather than individual keys or values.

## Required subclass hooks

### `mget`

Returns values corresponding to the supplied keys in the same order. Missing keys produce `None`.

```python
@abstractmethod
mget(
    self,
    keys: Sequence[K], # Keys whose values should be retrieved
) -> list[V | None] # Values corresponding to the supplied keys
```

### `mset`

Stores multiple key-value pairs.

```python
@abstractmethod
mset(
    self,
    key_value_pairs: Sequence[tuple[K, V]], # Key-value pairs to store
) -> None
```

### `mdelete`

Deletes multiple keys and their associated values.

```python
@abstractmethod
mdelete(
    self,
    keys: Sequence[K], # Keys to delete
) -> None
```

### `yield_keys`

Returns an iterator over keys matching an optional prefix.

```python
@abstractmethod
yield_keys(
    self,
    *,
    prefix: str | None = None, # Prefix used to filter keys
) -> Iterator[K] | Iterator[str] # Matching keys
```

The return type permits either the store key type `K` or strings, depending on the implementation.

## Optional subclass hooks

Native asynchronous stores can override `amget`, `amset`, `amdelete`, and `ayield_keys` instead of using the default executor-backed implementations.

## Methods

### `amget`

Runs `mget` through `run_in_executor`.

```python
async amget(
    self,
    keys: Sequence[K], # Keys whose values should be retrieved
) -> list[V | None] # Values corresponding to the supplied keys
```

### `amset`

Runs `mset` through `run_in_executor`.

```python
async amset(
    self,
    key_value_pairs: Sequence[tuple[K, V]], # Key-value pairs to store
) -> None
```

### `amdelete`

Runs `mdelete` through `run_in_executor`.

```python
async amdelete(
    self,
    keys: Sequence[K], # Keys to delete
) -> None
```

### `ayield_keys`

Obtains the synchronous iterator through `run_in_executor`, then retrieves each next item through the executor.

```python
async ayield_keys(
    self,
    *,
    prefix: str | None = None, # Prefix used to filter keys
) -> AsyncIterator[K] | AsyncIterator[str] # Matching keys
```

---

# `ByteStore`

Store interface with string keys and byte values.

```python
ByteStore = BaseStore[str, bytes]
```

---

# `InMemoryBaseStore: BaseStore[str, V], Generic[V]`

In-memory key-value store backed by a dictionary with string keys.

## Fields

```python
store: dict[str, V] # Underlying key-value dictionary
```

## Constructor

```python
InMemoryBaseStore(
    self,
) -> None
```

The constructor creates an empty dictionary.

## Methods

### `mget`

Returns stored values in input-key order, using `None` for missing keys.

```python
@override
mget(
    self,
    keys: Sequence[str], # Keys whose values should be retrieved
) -> list[V | None] # Values corresponding to the supplied keys
```

### `amget`

Calls `mget` directly without using an executor.

```python
@override
async amget(
    self,
    keys: Sequence[str], # Keys whose values should be retrieved
) -> list[V | None] # Values corresponding to the supplied keys
```

### `mset`

Stores or replaces each supplied key-value pair.

```python
@override
mset(
    self,
    key_value_pairs: Sequence[tuple[str, V]], # Key-value pairs to store
) -> None
```

### `amset`

Calls `mset` directly without using an executor.

```python
@override
async amset(
    self,
    key_value_pairs: Sequence[tuple[str, V]], # Key-value pairs to store
) -> None
```

### `mdelete`

Deletes each supplied key when present. Missing keys are ignored.

```python
@override
mdelete(
    self,
    keys: Sequence[str], # Keys to delete
) -> None
```

### `amdelete`

Calls `mdelete` directly without using an executor.

```python
@override
async amdelete(
    self,
    keys: Sequence[str], # Keys to delete
) -> None
```

### `yield_keys`

Yields all keys when `prefix` is `None`; otherwise yields keys whose strings start with the prefix.

```python
yield_keys(
    self,
    *,
    prefix: str | None = None, # Prefix used to filter keys
) -> Iterator[str] # Matching keys
```

Iteration follows the underlying dictionary's key order.

### `ayield_keys`

Asynchronously yields keys directly from the dictionary without using an executor.

```python
async ayield_keys(
    self,
    *,
    prefix: str | None = None, # Prefix used to filter keys
) -> AsyncIterator[str] # Matching keys
```

---

# `InMemoryStore: InMemoryBaseStore[Any]`

In-memory store for values of any type.

It inherits the dictionary-backed storage and all synchronous and asynchronous operations from `InMemoryBaseStore`.

---

# `InMemoryByteStore: InMemoryBaseStore[bytes]`

In-memory store for byte values.

It inherits the dictionary-backed storage and all synchronous and asynchronous operations from `InMemoryBaseStore`.

---

# `InvalidKeyException: LangChainException`

Raised when a store key is invalid, such as when it contains unsupported characters.

In [ ]:
from langchain_core.stores import InMemoryByteStore, InMemoryStore # Import the in-memory store classes
store = InMemoryStore() # Create an empty store that accepts values of any type

store.mset( # Store multiple key-value pairs
    [ # Begin the list of key-value pairs
        ("user:1", {"name": "Alice", "age": 25}), # Store the first user
        ("user:2", {"name": "Bob", "age": 30}), # Store the second user
        ("product:1", {"name": "Laptop", "price": 70000}), # Store a product
    ] # End the list of key-value pairs
) # Complete the batch storage operation

values = store.mget(["user:1", "user:2", "user:3"]) # Retrieve multiple keys in the given order

print(values) # Display stored values and None for the missing key



In [ ]:
# Cell 3: Filter and display keys
for key in store.yield_keys(prefix="user:"): # Iterate over keys beginning with user:
    print(key) # Display each matching key

In [ ]:
# Cell 4: Delete values
store.mdelete(["user:2", "missing:key"]) # Delete an existing key and ignore a missing key

remaining_values = store.mget(["user:1", "user:2"]) # Retrieve values after deletion

print(remaining_values) # Show that user:2 now returns None

In [ ]:
# Cell 5: Demonstrate InMemoryByteStore
byte_store = InMemoryByteStore() # Create an empty store for byte values

byte_store.mset( # Store multiple byte values
    [ # Begin the byte key-value pairs
        ("document:1", b"LangChain document content"), # Store the first byte sequence
        ("document:2", b"Another document"), # Store the second byte sequence
    ] # End the byte key-value pairs
) # Complete the batch storage operation

documents = byte_store.mget(["document:1", "document:2"]) # Retrieve the byte values

print(documents) # Display the raw byte values

decoded_document = documents[0].decode("utf-8") if documents[0] is not None else "" # Decode the first document safely

print(decoded_document) # Display the decoded document


In [ ]:
# Cell 6: Demonstrate asynchronous operations
async_store = InMemoryStore() # Create a separate store for asynchronous operations

await async_store.amset( # Store multiple values asynchronously
    [ # Begin the asynchronous key-value pairs
        ("session:1", "Active"), # Store the first session status
        ("session:2", "Inactive"), # Store the second session status
        ("config:theme", "Dark"), # Store a configuration value
    ] # End the asynchronous key-value pairs
) # Complete the asynchronous batch operation

sessions = await async_store.amget(["session:1", "session:2", "session:3"]) # Retrieve values asynchronously

print(sessions) # Display values and None for the missing session

In [ ]:
# Cell 7: Iterate asynchronously over keys
async for key in async_store.ayield_keys(prefix="session:"): # Iterate asynchronously over matching keys
    print(key) # Display each matching key



In [ ]:
Cell 8: Delete asynchronously
await async_store.amdelete(["session:1"]) # Delete the session key asynchronously

deleted_value = await async_store.amget(["session:1"]) # Retrieve the deleted key again

print(deleted_value) # Show that the deleted key now returns None